# 02 — Deploy as Hosted Agent

Deploy the Impossible Travel investigation workflow as a **Hosted Agent** in Microsoft Foundry.

This notebook packages the **same** workflow built locally in `01-investigation.ipynb`
(ContextEnricher → 10 risk sub-agents → RiskAggregator → PACO) into a container and
deploys it. The `test_cases/*.json` scenarios are bundled into the image so the
deployed agent can answer detections for any UPN found in those files.

### Deployment Lifecycle
1. **Package** — Generate `main.py`, `Dockerfile`, `requirements.txt`, and copy `test_cases/`
2. **Build** — `docker build` the container image (linux/amd64)
3. **Push** — Push to Azure Container Registry
4. **Deploy** — `project.agents.create_version()` registers the image with Foundry
5. **Invoke** — Send incidents via the Responses API

### Platform-Injected Environment Variables
| Variable | Description |
|---|---|
| `FOUNDRY_PROJECT_ENDPOINT` | Foundry project endpoint URL (auto-injected) |
| `FOUNDRY_AGENT_NAME` | Name of the running agent (auto-injected) |
| `APPLICATIONINSIGHTS_CONNECTION_STRING` | App Insights for telemetry (auto-injected) |
| `MODEL_DEPLOYMENT_NAME` | Deployment used by risk sub-agents (set during create_version) |
| `PACO_MODEL_DEPLOYMENT_NAME` | Deployment used by PACO; defaults to `MODEL_DEPLOYMENT_NAME` |

> **Prerequisites**: Run `00-setup.ipynb` first. Docker Desktop must be running.

In [9]:
# Load config

import json
import os
from pathlib import Path

config_path = Path("../workshop_config.json")
if not config_path.exists():
    raise FileNotFoundError("../workshop_config.json not found. Run 00-setup.ipynb first.")

with open(config_path) as f:
    config = json.load(f)

RESOURCE_GROUP = config["RESOURCE_GROUP"]
ACCOUNT_NAME = config["ACCOUNT_NAME"]
PROJECT_NAME = config["PROJECT_NAME"]
MODEL_DEPLOYMENT_NAME = config["MODEL_DEPLOYMENT_NAME"]
PROJECT_ENDPOINT = config["PROJECT_ENDPOINT"]

from azure.identity import AzureCliCredential
credential = AzureCliCredential()

print(f"✅ Config loaded — deploying to {ACCOUNT_NAME}/{PROJECT_NAME}")

✅ Config loaded — deploying to fndryiac2ttkx3/iac-ops-demo


## Generate Deployment Artifacts

In [10]:
# Sync the shared workflow module + test_cases/ into hosted_agent/ so they
# end up in the Docker build context. The hosted agent's main.py imports
# `investigation_workflow.py` directly — no MAIN_PY string duplication.

import shutil

HOSTED = Path("hosted_agent")
HOSTED.mkdir(exist_ok=True)

# 1) Shared workflow module (single source of truth, mirrored from repo root).
shared_src = Path("investigation_workflow.py")
shared_dst = HOSTED / "investigation_workflow.py"
shutil.copy2(shared_src, shared_dst)
print(f"  ✅ Synced: {shared_dst}")

# 2) Test case scenarios — the hosted agent loads them by UPN at startup.
src_cases = Path("test_cases")
dst_cases = HOSTED / "test_cases"
if dst_cases.exists():
    shutil.rmtree(dst_cases)
dst_cases.mkdir(parents=True, exist_ok=True)
case_count = 0
for case_file in sorted(src_cases.glob("*.json")):
    shutil.copy2(case_file, dst_cases / case_file.name)
    case_count += 1
print(f"  ✅ Bundled {case_count} test case(s) into {dst_cases}/")

# Sanity check that hosted_agent/main.py / Dockerfile / requirements.txt
# (checked-in files) are present in the build context.
for required in ("main.py", "Dockerfile", "requirements.txt"):
    p = HOSTED / required
    if not p.exists():
        raise FileNotFoundError(f"Expected {p} to exist (committed to the repo).")
    print(f"  ✓ Present: {p}")

# Enforce store=True in the bundled hosted_agent/main.py so Foundry persists
# session/thread metadata. Without this the portal cannot show full session
# details, token counts, or traces for the deployed agent.
hosted_main = HOSTED / "main.py"
main_text = hosted_main.read_text(encoding="utf-8")
if "store=False" in main_text:
    hosted_main.write_text(main_text.replace("store=False", "store=True"), encoding="utf-8")
    print("  ✅ Updated hosted_agent/main.py to store=True for portal session visibility")
elif "store=True" in main_text:
    print("  ✓ hosted_agent/main.py already uses store=True")
else:
    print("  ⚠️  Could not find store flag in hosted_agent/main.py; verify manually.")

print("\n📁 Deployment artifacts ready in hosted_agent/")

  ✅ Synced: hosted_agent\investigation_workflow.py
  ✅ Bundled 5 test case(s) into hosted_agent\test_cases/
  ✓ Present: hosted_agent\main.py
  ✓ Present: hosted_agent\Dockerfile
  ✓ Present: hosted_agent\requirements.txt
  ⚠️  Could not find store flag in hosted_agent/main.py; verify manually.

📁 Deployment artifacts ready in hosted_agent/


## Build, Push & Deploy

In [11]:
import asyncio
import subprocess
import sys

AGENT_NAME = "impossible-travel-investigator"
IMAGE_TAG = "v1"

# Optional manual override — set this if auto-discovery below can't find your ACR
# (e.g. your registry lives in a different resource group / subscription).
#   ACR_LOGIN_SERVER = "myregistry.azurecr.io"
ACR_LOGIN_SERVER = ""

# On Windows, `az` and `docker` are shipped as .cmd shims that CreateProcess
# can't launch directly — use shell=True so cmd.exe resolves them via PATHEXT.
_USE_SHELL = sys.platform == "win32"


def _run_az(args: list[str]) -> str:
    """Run an `az` command and return stdout. Print stderr on failure for diagnostics."""
    result = subprocess.run(
        ["az", *args], capture_output=True, text=True, shell=_USE_SHELL,
    )
    if result.returncode != 0:
        print(f"  ⚠️  az {' '.join(args)} (exit {result.returncode})")
        if result.stderr.strip():
            print(f"     stderr: {result.stderr.strip().splitlines()[0][:200]}")
        return ""
    return result.stdout.strip()


# ── Preflight: Docker daemon must be running ──────────────────
_docker_check = subprocess.run(
    ["docker", "info"], capture_output=True, text=True, shell=_USE_SHELL,
)
if _docker_check.returncode != 0:
    raise RuntimeError(
        "Docker daemon is not reachable. Start Docker Desktop and wait for it to be ready, "
        "then re-run this cell.\n"
        f"  docker info stderr: {_docker_check.stderr.strip().splitlines()[0][:200] if _docker_check.stderr.strip() else '(none)'}"
    )


# ── Step 1: Discover ACR ──────────────────────────────────────
print("▶ Step 1: Discovering Azure Container Registry...")

if not ACR_LOGIN_SERVER:
    # 1a. Some Foundry/AI Services resources expose a linked ACR via properties.containerRegistry.
    acr_id = _run_az([
        "cognitiveservices", "account", "show",
        "--name", ACCOUNT_NAME, "--resource-group", RESOURCE_GROUP,
        "--query", "properties.containerRegistry", "-o", "tsv",
    ])
    if acr_id and acr_id != "None":
        acr_name = acr_id.split("/")[-1] if "/" in acr_id else acr_id
        ACR_LOGIN_SERVER = _run_az([
            "acr", "show", "--name", acr_name, "--query", "loginServer", "-o", "tsv",
        ])
        if ACR_LOGIN_SERVER:
            print(f"  Found ACR linked to Foundry resource: {ACR_LOGIN_SERVER}")

if not ACR_LOGIN_SERVER:
    # 1b. Fall back to the first ACR in the same resource group.
    ACR_LOGIN_SERVER = _run_az([
        "acr", "list", "--resource-group", RESOURCE_GROUP,
        "--query", "[0].loginServer", "-o", "tsv",
    ])
    if ACR_LOGIN_SERVER:
        print(f"  Found ACR in resource group '{RESOURCE_GROUP}': {ACR_LOGIN_SERVER}")

if not ACR_LOGIN_SERVER:
    # 1c. Last resort — scan the entire subscription.
    ACR_LOGIN_SERVER = _run_az([
        "acr", "list", "--query", "[0].loginServer", "-o", "tsv",
    ])
    if ACR_LOGIN_SERVER:
        print(f"  Found ACR in subscription (first match): {ACR_LOGIN_SERVER}")

if not ACR_LOGIN_SERVER:
    raise RuntimeError(
        "Could not discover an Azure Container Registry.\n"
        "Either:\n"
        f"  • Create one:  az acr create -g {RESOURCE_GROUP} -n <name> --sku Basic --admin-enabled true\n"
        "  • Or set ACR_LOGIN_SERVER manually at the top of this cell (e.g. 'myregistry.azurecr.io')."
    )

FULL_IMAGE = f"{ACR_LOGIN_SERVER}/{AGENT_NAME}:{IMAGE_TAG}"
print(f"  ACR:   {ACR_LOGIN_SERVER}")
print(f"  Image: {FULL_IMAGE}")

# ── Step 2: Build ─────────────────────────────────────────────
print("\n▶ Step 2: Building Docker image...")
subprocess.run(
    ["docker", "build", "--platform", "linux/amd64", "-t", f"{AGENT_NAME}:{IMAGE_TAG}", "hosted_agent/"],
    check=True, shell=_USE_SHELL,
)
print("  ✅ Image built")

# ── Step 3: Push ──────────────────────────────────────────────
print("\n▶ Step 3: Pushing to ACR...")
acr_name = ACR_LOGIN_SERVER.split(".")[0]
subprocess.run(["az", "acr", "login", "--name", acr_name], check=True, shell=_USE_SHELL)
subprocess.run(["docker", "tag", f"{AGENT_NAME}:{IMAGE_TAG}", FULL_IMAGE], check=True, shell=_USE_SHELL)
subprocess.run(["docker", "push", FULL_IMAGE], check=True, shell=_USE_SHELL)
print("  ✅ Image pushed")

# ── Step 3.5: Ensure RBAC for image pull + model access ───────
# Foundry's manual deploy path requires three role assignments that the
# `azd` / VS Code Toolkit paths grant automatically. Without these the version
# either fails to pull the image or returns 500 at invoke time.
#
#   (a) AcrPull on the ACR  →  Foundry PROJECT's system-assigned managed identity
#                              (Foundry uses this to pull the image)
#   (b) Cognitive Services OpenAI User on the Foundry account
#                          →  the per-agent runtime identity
#                              (container calls Foundry chat endpoints via this)
#   (c) Foundry User on the Foundry account
#                          →  the per-agent runtime identity
#                              (formerly "Azure AI User"; required at runtime)
print("\n▶ Step 3.5: Ensuring required RBAC role assignments...")

_acr_resource_id = _run_az([
    "acr", "show", "--name", acr_name, "--query", "id", "-o", "tsv",
])
_foundry_account_id = _run_az([
    "cognitiveservices", "account", "show",
    "--name", ACCOUNT_NAME, "--resource-group", RESOURCE_GROUP,
    "--query", "id", "-o", "tsv",
])
_project_principal = _run_az([
    "resource", "show",
    "--ids", f"{_foundry_account_id}/projects/{PROJECT_NAME}",
    "--query", "identity.principalId", "-o", "tsv",
])


def _ensure_app_insights() -> tuple[str, str]:
    """Ensure an Application Insights resource exists in RESOURCE_GROUP.

    Returns ``(resource_id, connection_string)``. Strategy:

    1. Try to discover an App Insights already linked to the Foundry
       account/project (legacy property names, kept as a fast path).
    2. Otherwise look for any App Insights component in the resource group.
    3. Otherwise provision a fresh workspace-based App Insights
       (``<account>-appi``) backed by a Log Analytics workspace
       (``<account>-law``) in the same region.
    """
    appi_name = f"{ACCOUNT_NAME}-appi"
    law_name = f"{ACCOUNT_NAME}-law"
    location = _run_az([
        "group", "show", "-n", RESOURCE_GROUP, "--query", "location", "-o", "tsv",
    ]) or "eastus"

    for q in (
        "properties.applicationInsightsResourceId",
        "properties.applicationInsightsId",
        "properties.applicationInsights",
        "properties.monitoring.applicationInsightsResourceId",
        "properties.monitoring.appInsightsResourceId",
    ):
        rid = _run_az([
            "cognitiveservices", "account", "show",
            "--name", ACCOUNT_NAME, "--resource-group", RESOURCE_GROUP,
            "--query", q, "-o", "tsv",
        ])
        if rid and rid != "None" and rid.startswith("/subscriptions/"):
            print(f"  • Reusing App Insights linked on Foundry account: {rid.split('/')[-1]}")
            conn = _run_az([
                "monitor", "app-insights", "component", "show",
                "--ids", rid, "--query", "connectionString", "-o", "tsv",
            ])
            return rid, conn

    rid = _run_az([
        "resource", "list",
        "--resource-group", RESOURCE_GROUP,
        "--resource-type", "microsoft.insights/components",
        "--query", "[0].id", "-o", "tsv",
    ])
    if rid and rid != "None":
        print(f"  • Reusing App Insights in resource group: {rid.split('/')[-1]}")
        conn = _run_az([
            "monitor", "app-insights", "component", "show",
            "--ids", rid, "--query", "connectionString", "-o", "tsv",
        ])
        return rid, conn

    law_id = _run_az([
        "monitor", "log-analytics", "workspace", "show",
        "-g", RESOURCE_GROUP, "-n", law_name,
        "--query", "id", "-o", "tsv",
    ])
    if not law_id:
        print(f"  • Creating Log Analytics workspace '{law_name}' in {location}...")
        law_id = _run_az([
            "monitor", "log-analytics", "workspace", "create",
            "-g", RESOURCE_GROUP, "-n", law_name,
            "--location", location,
            "--query", "id", "-o", "tsv",
        ])

    print(f"  • Creating Application Insights '{appi_name}' in {location}...")
    appi_id = _run_az([
        "monitor", "app-insights", "component", "create",
        "-g", RESOURCE_GROUP, "-a", appi_name,
        "--location", location,
        "--workspace", law_id,
        "--query", "id", "-o", "tsv",
    ])
    conn = _run_az([
        "monitor", "app-insights", "component", "show",
        "-g", RESOURCE_GROUP, "-a", appi_name,
        "--query", "connectionString", "-o", "tsv",
    ])
    return appi_id, conn


_appi_resource_id, _appi_connection_string = _ensure_app_insights()
if _appi_resource_id and _appi_connection_string:
    print(f"  ✓ App Insights:  {_appi_resource_id.split('/')[-1]}")
    print(f"  ✓ Connection string captured (length={len(_appi_connection_string)})")
else:
    print("  ⚠️  Could not provision/discover App Insights — telemetry will be disabled.")


def _ensure_role(principal_id: str, role: str, scope: str, label: str) -> None:
    if not principal_id or principal_id == "None":
        print(f"  ⚠️  Skipping '{role}' for {label}: no principal id resolved.")
        return
    existing = _run_az([
        "role", "assignment", "list",
        "--assignee", principal_id, "--scope", scope,
        "--query", f"[?roleDefinitionName=='{role}'] | length(@)", "-o", "tsv",
    ])
    if existing and existing != "0":
        print(f"  ✓ '{role}' already granted to {label} on {scope.split('/')[-1]}")
        return
    print(f"  • Granting '{role}' to {label} ({principal_id[:8]}…) on {scope.split('/')[-1]}")
    _run_az([
        "role", "assignment", "create",
        "--assignee-object-id", principal_id,
        "--assignee-principal-type", "ServicePrincipal",
        "--role", role, "--scope", scope,
    ])


# (a) Image pull
_ensure_role(_project_principal, "AcrPull", _acr_resource_id, "Foundry project MI")

# Determine the per-agent runtime identity (created by Foundry when a version exists).
# On the very first deploy this won't exist yet — we'll re-run after create_version.
try:
    _existing_agent = project_client.agents.get(agent_name=AGENT_NAME).as_dict()
    _runtime_principal = (_existing_agent.get("instance_identity") or {}).get("principal_id")
except Exception:
    _runtime_principal = None

if _runtime_principal:
    _ensure_role(_runtime_principal, "Cognitive Services OpenAI User", _foundry_account_id, "agent runtime MI")
    _ensure_role(_runtime_principal, "Foundry User", _foundry_account_id, "agent runtime MI")
    if _appi_resource_id:
        _ensure_role(_runtime_principal, "Monitoring Metrics Publisher", _appi_resource_id, "agent runtime MI")
else:
    print("  ℹ️  Per-agent runtime identity not yet created — will be granted after first create_version.")

# Best-effort: attach the App Insights to the Foundry project so the portal
# "Tracing" view knows to read from it AND so Foundry auto-injects the matching
# APPLICATIONINSIGHTS_CONNECTION_STRING into the container (it reserves that
# env var name, we can't set it ourselves). Property name + api-version vary
# across Foundry releases; try a few combinations.
if _appi_resource_id:
    _project_resource_id = f"{_foundry_account_id}/projects/{PROJECT_NAME}"
    _attached = False
    _attach_attempts = [
        ("properties.applicationInsightsResourceId", "2025-06-01"),
        ("properties.applicationInsightsResourceId", "2024-10-01-preview"),
        ("properties.applicationInsightsResourceId", None),
        ("properties.applicationInsightsId", None),
        ("properties.applicationInsights", None),
    ]
    for _prop, _api in _attach_attempts:
        _cmd = [
            "resource", "update", "--ids", _project_resource_id,
            "--set", f"{_prop}={_appi_resource_id}",
            "--query", "id", "-o", "tsv",
        ]
        if _api:
            _cmd.extend(["--api-version", _api])
        if _run_az(_cmd):
            print(f"  ✓ Linked App Insights to Foundry project via {_prop}"
                  + (f" (api-version {_api})" if _api else ""))
            _attached = True
            break
    if not _attached:
        print("  ℹ️  Could not auto-link App Insights at the project level.")
        print("     Manual fix: in the Foundry Portal, open the project →")
        print("     'Tracing' (or 'Application Insights') → 'Connect resource'")
        print(f"     and pick: {_appi_resource_id.split('/')[-1]}")
        print("     After attaching, redeploy this cell so the new agent version")
        print("     picks up the platform-injected connection string.")

# ── Step 4: Create Hosted Agent ───────────────────────────────
print("\n▶ Step 4: Creating Hosted Agent version...")

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True,
)

# Pass the model deployment name to the container. PACO_MODEL_DEPLOYMENT_NAME is
# optional — set it to a different deployment if you want PACO to reason with a
# stronger model than the risk evaluators (matches the pattern in 01-investigation.ipynb).
PACO_MODEL_DEPLOYMENT_NAME = os.environ.get("PACO_MODEL_DEPLOYMENT_NAME", MODEL_DEPLOYMENT_NAME)

# Override the platform's default APPLICATIONINSIGHTS_CONNECTION_STRING with the
# one we just provisioned so the OTEL exporter has a target it can authenticate
# against (the runtime MI now has 'Monitoring Metrics Publisher' on this resource).
# NOTE: Foundry reserves APPLICATIONINSIGHTS_CONNECTION_STRING (along with all
# FOUNDRY_* and AGENT_* names) and rejects requests that try to set it via
# environment_variables. The platform injects its own value at runtime once an
# App Insights resource is attached to the project (handled above). The runtime
# MI was granted Monitoring Metrics Publisher on that App Insights, so telemetry
# will flow as soon as the platform-injected connection string points there.
_agent_env_vars: dict[str, str] = {
    "MODEL_DEPLOYMENT_NAME": str(MODEL_DEPLOYMENT_NAME),
    "PACO_MODEL_DEPLOYMENT_NAME": str(PACO_MODEL_DEPLOYMENT_NAME),
}

agent_version = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="1.0.0")
        ],
        cpu="1", memory="2Gi", image=FULL_IMAGE,
        environment_variables=_agent_env_vars,
    ),
)
print(f"  Agent: {agent_version.name}, Version: {agent_version.version}")

# ── Step 4.5: Grant runtime model-access roles ────────────────
# After the first create_version, Foundry has provisioned the per-agent runtime
# identity. Grant the runtime roles now (idempotent on subsequent deploys).
try:
    _agent_doc = project_client.agents.get(agent_name=AGENT_NAME).as_dict()
    _runtime_principal = (_agent_doc.get("instance_identity") or {}).get("principal_id")
    if _runtime_principal:
        _ensure_role(_runtime_principal, "Cognitive Services OpenAI User", _foundry_account_id, "agent runtime MI")
        _ensure_role(_runtime_principal, "Foundry User", _foundry_account_id, "agent runtime MI")
        if _appi_resource_id:
            _ensure_role(_runtime_principal, "Monitoring Metrics Publisher", _appi_resource_id, "agent runtime MI")
        else:
            print("  ⚠️  Skipping App Insights RBAC: no resource id available.")
        print("  ℹ️  Runtime RBAC propagation can take 30–90s before the agent can call models.")
except Exception as _exc:
    print(f"  ⚠️  Could not resolve runtime identity after create_version: {_exc}")

# ── Step 5: Poll for active ───────────────────────────────────
print("\n▶ Step 5: Waiting for agent to become active...")
while True:
    version_info = project_client.agents.get_version(
        agent_name=AGENT_NAME, agent_version=agent_version.version,
    )
    status = version_info["status"]
    print(f"  Status: {status}")
    if status == "active":
        print("  ✅ Agent is ready!")
        break
    elif status == "failed":
        print(f"  ❌ Provisioning failed: {version_info.get('error')}")
        break
    await asyncio.sleep(5)


▶ Step 1: Discovering Azure Container Registry...
  Found ACR in subscription (first match): htamlcr.azurecr.io
  ACR:   htamlcr.azurecr.io
  Image: htamlcr.azurecr.io/impossible-travel-investigator:v1

▶ Step 2: Building Docker image...
  ✅ Image built

▶ Step 3: Pushing to ACR...
  ✅ Image pushed

▶ Step 3.5: Ensuring required RBAC role assignments...
  • Reusing App Insights in resource group: fndryiac2ttkx3-appi
  ✓ App Insights:  fndryiac2ttkx3-appi
  ✓ Connection string captured (length=252)
  ✓ 'AcrPull' already granted to Foundry project MI on htamlcr
  ✓ 'Cognitive Services OpenAI User' already granted to agent runtime MI on fndryiac2ttkx3
  ✓ 'Foundry User' already granted to agent runtime MI on fndryiac2ttkx3
  ✓ 'Monitoring Metrics Publisher' already granted to agent runtime MI on fndryiac2ttkx3-appi
  ✓ Linked App Insights to Foundry project via properties.applicationInsightsResourceId (api-version 2025-06-01)

▶ Step 4: Creating Hosted Agent version...
  Agent: impossible

## Invoke the Deployed Agent

In [12]:
from pydantic import BaseModel, Field
from typing import Any

# Re-define the InvestigationVerdict shape (matches 01-investigation.ipynb)
# so we can parse and pretty-print the deployed agent's structured response.
class ActionItem(BaseModel):
    action: str
    target: str = ""
    value: str = ""
    riskFactor: str = ""
    destructive: bool = False
    requiresApproval: bool = False

class InvestigationVerdict(BaseModel):
    verdict: str
    confidence: str
    reasoning: str
    actionPlan: list[ActionItem] = Field(default_factory=list)
    narrative: str = ""


# Load the same test case the local notebook uses (must be one of the JSON
# scenarios bundled into the container under hosted_agent/test_cases/).
# Change TEST_CASE to exercise a different scenario.
from pathlib import Path

TEST_CASE = "03_mfa_fatigue_attack"
case_path = Path("test_cases") / f"{TEST_CASE}.json"
with open(case_path, encoding="utf-8") as f:
    test_case_data = json.load(f)

detection_payload = test_case_data["detection"]
incident_json = json.dumps(detection_payload)

print(f"▶ Invoking deployed agent with scenario: {test_case_data['name']}")
print(f"   User:      {detection_payload['PrimaryEntityId']}")
print(f"   Expected:  {test_case_data.get('expected_verdict', '—')}"
      f" ({test_case_data.get('expected_confidence', '—')} confidence)")

openai_client = project_client.get_openai_client(agent_name=AGENT_NAME)
response = openai_client.responses.create(input=incident_json)

print("\n" + "═" * 70)
print("📋 HOSTED AGENT RESPONSE")
print("═" * 70)
print(response.output_text)

try:
    hosted_verdict = InvestigationVerdict.model_validate_json(response.output_text)
    print(f"\n  Verdict:    {hosted_verdict.verdict}")
    print(f"  Confidence: {hosted_verdict.confidence}")
    print(f"  Actions:    {len(hosted_verdict.actionPlan)}")
except Exception:
    print("  (Could not parse structured verdict from response)")

▶ Invoking deployed agent with scenario: MFA Fatigue Attack
   User:      bob.chen@contoso.com
   Expected:  AccountCompromised (High confidence)

══════════════════════════════════════════════════════════════════════
📋 HOSTED AGENT RESPONSE
══════════════════════════════════════════════════════════════════════
{
  "verdict": "AccountCompromised",
  "confidence": "High",
  "reasoning": "Multiple strong risk indicators converge indicating a high likelihood of account compromise for user bob.chen@contoso.com. A suspicious geographic login anomaly from a known proxy IP in Sofia, BG far from usual Seattle location is detected with a high evidence score (85). Concurrently, there's a pattern of brute force attempts followed by successful login from the same suspicious IP (score 85), device compliance failures (score 65), and anomalous token usage across multiple apps (score 75). Notably, MFA fatigue attacks are evident with multiple denied then accepted prompts and an immediate suspicious MF

## Telemetry Diagnostic

If the Foundry Portal **Tracing** / **Sessions** pane is empty after a redeploy,
run this cell to find out **which Application Insights resource Foundry expects
telemetry to land in** and whether the agent runtime identity actually has
`Monitoring Metrics Publisher` on that exact resource. Mismatches between the
auto-injected `APPLICATIONINSIGHTS_CONNECTION_STRING` and the resource that
got the RBAC grant are the most common cause of missing traces.

## Token Usage Report

Query Application Insights to get the **actual** per-agent token consumption for the
most recent invocation. The Foundry Tracing view may only show a subset of the
10 sub-agent spans in the timeline; this KQL query captures all of them.

Run this cell after the invoke cell above. It waits briefly for ingestion
then prints a table with input/output tokens per agent and a grand total.

In [ ]:
# Token usage report — queries App Insights for accurate per-agent token counts.
# The Foundry Tracing timeline may only show a subset of the 10 sub-agent spans;
# this query surfaces them all.
#
# This cell is self-contained: it re-resolves the App Insights resource and
# the az CLI helper so it works even after a kernel restart without re-running
# the full deploy cell.

import subprocess as _subprocess
import sys as _sys
import time as _time
import json as _json
from pathlib import Path as _Path

# Re-read config if not already in scope
if "RESOURCE_GROUP" not in dir():
    _cfg_path = _Path("workshop_config.json")
    if not _cfg_path.exists():
        _cfg_path = _Path("../workshop_config.json")
    with open(_cfg_path) as _f:
        _cfg = _json.load(_f)
    RESOURCE_GROUP = _cfg["RESOURCE_GROUP"]
    ACCOUNT_NAME = _cfg["ACCOUNT_NAME"]

_USE_SHELL_TOKEN = _sys.platform == "win32"


def _run_az_token(args: list[str]) -> str:
    r = _subprocess.run(["az", *args], capture_output=True, text=True, shell=_USE_SHELL_TOKEN)
    return r.stdout.strip() if r.returncode == 0 else ""


# Discover App Insights in the resource group
_appi_rid = _run_az_token([
    "resource", "list",
    "--resource-group", RESOURCE_GROUP,
    "--resource-type", "microsoft.insights/components",
    "--query", "[0].id", "-o", "tsv",
])
_appi_name = _appi_rid.split("/")[-1] if _appi_rid and _appi_rid != "None" else ""

# How many seconds to wait for App Insights ingestion (telemetry has ~30-60s delay).
INGESTION_WAIT_S = 45
LOOKBACK_MINUTES = 30

print(f"App Insights: {_appi_name or '(not found)'}")
AGENT_FILTER = 'impossible-travel-investigator'
print(f'Agent filter: {AGENT_FILTER}')
print(f"Waiting {INGESTION_WAIT_S}s for App Insights ingestion...")
_time.sleep(INGESTION_WAIT_S)

# Query via az monitor app-insights query
_kql = (
    'dependencies'
    '| where timestamp > ago(' + str(LOOKBACK_MINUTES) + 'm)'
    '| where name startswith "chat "'
    '| extend agent_id = tostring(coalesce(customDimensions["gen_ai.agent.name"], customDimensions["gen_ai.agent.id"]))'
    '| where agent_id has "' + AGENT_FILTER + '"'
    '| extend'
    '    agent  = tostring(coalesce(customDimensions["gen_ai.agent.name"], customDimensions["gen_ai.agent.id"], name)),'
    '    in_t   = toint(coalesce(customDimensions["gen_ai.usage.input_tokens"],  customDimensions["gen_ai.usage.prompt_tokens"], customDimensions["llm.usage.prompt_tokens"])),'
    '    out_t  = toint(coalesce(customDimensions["gen_ai.usage.output_tokens"], customDimensions["gen_ai.usage.completion_tokens"], customDimensions["llm.usage.completion_tokens"])),'
    '    model  = tostring(coalesce(customDimensions["gen_ai.response.model"],   customDimensions["llm.response.model"], name))'
    '| where isnotnull(in_t) or isnotnull(out_t)'
    '| summarize calls = count(),'
    '            input_tokens  = sum(in_t),'
    '            output_tokens = sum(out_t),'
    '            total_tokens  = sum(in_t) + sum(out_t)'
    '  by agent, model'
    '| order by agent asc'
)
if not _appi_name:
    print('⚠️  No App Insights resource id available. Run the deploy cell first.')
else:
    _raw = _run_az_token([
        'monitor', 'app-insights', 'query',
        '--app', _appi_name,
        '-g', RESOURCE_GROUP,
        '--analytics-query', _kql,
        '-o', 'json',
    ])
    if not _raw:
        print('⚠️  Query returned no data. Either no invocations in the last'
              f' {LOOKBACK_MINUTES} minutes, or telemetry has not been ingested yet.')
        print('    Try increasing LOOKBACK_MINUTES or re-running after a fresh invocation.')
    else:
        _result = _json.loads(_raw)
        _tables = _result.get('tables', _result) if isinstance(_result, dict) else _result
        if isinstance(_tables, list) and len(_tables) > 0:
            _table = _tables[0] if isinstance(_tables[0], dict) else _tables
            _columns = _table.get('columns', []) if isinstance(_table, dict) else []
            _rows = _table.get('rows', []) if isinstance(_table, dict) else []
            _col_names = [c.get('name', c) if isinstance(c, dict) else str(c) for c in _columns]

            print(f"{'Agent':<40} {'Model':<18} {'Calls':>5} {'Input':>8} {'Output':>8} {'Total':>8}")
            print('─' * 96)
            grand_in = grand_out = grand_total = grand_calls = 0
            for row in _rows:
                agent   = str(row[_col_names.index('agent')]   if 'agent'   in _col_names else '')
                model   = str(row[_col_names.index('model')]   if 'model'   in _col_names else '')
                calls   = int(row[_col_names.index('calls')]   if 'calls'   in _col_names else 0)
                in_t    = int(row[_col_names.index('input_tokens')]  if 'input_tokens'  in _col_names else 0)
                out_t   = int(row[_col_names.index('output_tokens')] if 'output_tokens' in _col_names else 0)
                total_t = int(row[_col_names.index('total_tokens')]  if 'total_tokens'  in _col_names else 0)
                print(f'{agent:<40} {model:<18} {calls:>5} {in_t:>8} {out_t:>8} {total_t:>8}')
                grand_in += in_t; grand_out += out_t; grand_total += total_t; grand_calls += calls
            print('─' * 96)
            print(f"{'TOTAL':<40} {'':<18} {grand_calls:>5} {grand_in:>8} {grand_out:>8} {grand_total:>8}")
        else:
            print('⚠️  Unexpected query result shape. Raw response:')
            print(_json.dumps(_result, indent=2)[:2000])

In [14]:
# Cleanup (Optional)
# Uncomment the lines below to delete the hosted agent.

# print("▶ Cleaning up hosted agent...")
# project_client.agents.delete(agent_name=AGENT_NAME)
# print("✅ Agent deleted")

## Architecture & Production Notes

### SOC Document → Agent Framework Mapping

| SOC Document Concept | Agent Framework Implementation |
|---|---|
| Sentinel Analytics Rule | `NormalizedDetection` — input trigger |
| Context Agent | `ContextEnricher(Executor)` — deterministic, no LLM |
| Identity Agent (10 tools) | 10 specialized `Agent` instances with fan-out |
| RiskEvidence generation | Structured output via `response_format=RiskEvidence` |
| ThreatDecisionRecord | `RiskAggregator(Executor)` — deterministic assembly |
| ThreatDecisionPolicy | Embedded in PACO's system prompt |
| PACO decision maker | `Agent` with `response_format=InvestigationVerdict` |
| ActionPlan generation | Part of PACO's structured output |
| Pipeline orchestration | `WorkflowBuilder` with fan-out/fan-in graph |

### Production Considerations

1. **Real Data Sources via MCP**: Replace `@tool` simulated data with MCP connections to:
   - Microsoft Sentinel (KQL queries for sign-in logs, audit logs, alerts)
   - Microsoft Entra ID (user profiles, roles, groups)
   - Microsoft Defender XDR (incident correlation)

2. **Human-in-the-Loop**: Use `ctx.request_info()` in the workflow
   to pause and await analyst approval before executing destructive actions on privileged users.

3. **Tracing & Observability**: Hosted Agents auto-inject Application Insights.
   View investigation traces in Foundry Portal → Tracing.

4. **Scaling**: Each Sentinel detection triggers one workflow run.
   Foundry Hosted Agents scale-to-zero with 15-minute idle timeout.